**Classical model**

This notebook forms the layout for the first of the two models: the classical one. In both models, we're going to use the "California Housing Prices: Median house prices for California districts derived from the 1990 census" as it's well known and presents interesting challenges in cleaning the data to be suitable for both types of models. It's also used very frequently in learning machine learning. We start by importing important libraries and fetching the data set:

In [11]:
import numpy as np, json, time
from pathlib import Path
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
import pandas as pd

Numpy is for numrical operations, sklearn for tuning, cleaning, and classifying the data (the real deal), and pandas is for dealing with datasets. Now we take a look at the data:

In [14]:
data = fetch_california_housing(as_frame= True)
df = data.frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


To simplify everything, we choose just 4 features of the dataset to work with, which are the most important ones to predict. The job of the classifier will be predicting wether the house price is above or below the median of all houses, where 1 indicates above and 0 indicates below. We will start by labeling the data:

In [15]:
FEATURES = ["MedInc", "Latitude", "Longitude", "HouseAge"]
y_all = (df["MedHouseVal"] > df["MedHouseVal"].median()).astype(int).values
X_all = df[FEATURES].values
print(X_all.shape, y_all.shape)
print("class balance:", np.bincount(y_all))

(20640, 4) (20640,)
class balance: [10323 10317]


The class balance is roughly 50/50, which is great. As the data is about 20k rows and the quantum model is too slow for that, we'll use 800 of the rows. We start by choosing these rows and stratisfying them so that they're also roughly 50/50 on class balance:

In [16]:
SEED = 0
N_SAMPLES = 800
X_sub, _, y_sub, _ = train_test_split(X_all, y_all, train_size=N_SAMPLES, stratify=y_all, random_state=SEED)
print(X_sub.shape, np.bincount(y_sub))

(800, 4) [400 400]


Now we split our training and test split normally:

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X_sub, y_sub, test_size=0.25, stratify=y_sub, random_state=SEED)
print("train:", X_train.shape, np.bincount(y_train))
print("test: ", X_test.shape, np.bincount(y_test))

train: (600, 4) [300 300]
test:  (200, 4) [100 100]


Now we standardize and transform the data into [0, pi] range to be able to use it in the quantum model. We use QuantileTransformer in order to gice each entry a number between 0 and 1 first, then we multiply by required range [0, pi]:

In [ ]:
qt = QuantileTransformer(n_quantiles=200, output_distribution="uniform")
qt.fit(X_train)                                   # train only
X_train, X_test = qt.transform(X_train) * np.pi, qt.transform(X_test) * np.pi
np.savez(f"splits/seed{seed}.npz", Xtr=X_train, Xte=X_test, ytr=y_train, yte=y_test)